In [ ]:
"""Tải XAUUSD M1 từ MetaTrader 5 (chỉ chạy trên Windows, MT5 phải đã đăng nhập),
lưu Parquet và in báo cáo chất lượng dữ liệu.

Cài đặt: pip install MetaTrader5 pandas pyarrow
"""
from datetime import datetime, timezone
from pathlib import Path

import MetaTrader5 as mt5
import pandas as pd

SYMBOL = "GOLD"  # tên có thể khác tùy broker: GOLD, XAUUSDm, XAUUSD.a ...
START = datetime(2001, 1, 1, tzinfo=timezone.utc)
END = datetime.now(timezone.utc)
OUT = Path("data")
OUT.mkdir(exist_ok=True)


def download() -> pd.DataFrame:
    if not mt5.initialize():
        raise RuntimeError(f"MT5 initialize lỗi: {mt5.last_error()}")
    if not mt5.symbol_select(SYMBOL, True):
        raise RuntimeError(f"Không tìm thấy symbol {SYMBOL}, kiểm tra tên ở Market Watch")

    frames = []
    for month_start in pd.date_range(START, END, freq="MS", tz="UTC"):
        month_end = min(month_start + pd.offsets.MonthBegin(1), pd.Timestamp(END))
        rates = mt5.copy_rates_range(
            SYMBOL, mt5.TIMEFRAME_M1,
            month_start.to_pydatetime(), month_end.to_pydatetime(),
        )
        n = 0 if rates is None else len(rates)
        print(f"{month_start:%Y-%m}: {n} nến")
        if n:
            frames.append(pd.DataFrame(rates))
    mt5.shutdown()

    df = pd.concat(frames, ignore_index=True)
    # LƯU Ý: đây là giờ SERVER của broker (không phải UTC), cần xác định offset sau
    df["time"] = pd.to_datetime(df["time"], unit="s")
    df = df.drop_duplicates("time").sort_values("time").set_index("time")
    return df[["open", "high", "low", "close", "tick_volume", "spread"]]


def quality_report(df: pd.DataFrame) -> None:
    print("\n=== BÁO CÁO CHẤT LƯỢNG ===")
    print(f"Số nến: {len(df):,} | từ {df.index[0]} đến {df.index[-1]}")
    print(f"Nến lỗi high<low: {(df['high'] < df['low']).sum()}")
    print(f"Nến có giá <= 0: {(df[['open','high','low','close']] <= 0).any(axis=1).sum()}")

    gaps = df.index.to_series().diff().dropna()
    big = gaps[gaps > pd.Timedelta(minutes=5)]
    # bỏ qua khoảng nghỉ cuối tuần (> 40 giờ)
    big = big[big < pd.Timedelta(hours=40)]
    print(f"Khoảng trống 5 phút đến 40 giờ (ngoài cuối tuần): {len(big)}")
    print(big.sort_values(ascending=False).head(10))

    print("\nSpread (points):")
    print(df["spread"].describe(percentiles=[0.5, 0.9, 0.99]))
    print("\nSố nến theo giờ trong ngày (để đoán offset múi giờ broker):")
    print(df.groupby(df.index.hour).size())


if __name__ == "__main__":
    data = download()
    data.to_parquet(OUT / f"{SYMBOL}_M1.parquet")
    quality_report(data)

In [ ]:
"""Làm sạch XAUUSD M1 từ MT5 và dựng kho dữ liệu (Parquet + DuckDB).

Đầu vào : data/XAUUSD_M1.parquet  (đầu ra của get_data.py, giờ SERVER broker)
Đầu ra  : data/raw/mt5/XAUUSD_M1/<sha8>/year=*/   bản gốc, bất biến (đặt tên theo hash file)
          data/clean/XAUUSD_M1_utc/year=*/        M1 đã cắt nến thô + đổi sang UTC
          data/derived/XAUUSD_H1_utc/year=*/      H1 dựng từ M1 sạch
          data/meta.duckdb                        bảng nguồn + view SQL (m1_utc, h1_utc)
          config/splits.json                      khoảng ngày dev / bền vững / holdout (chỉ tạo nếu chưa có)

Cài đặt: pip install pandas pyarrow duckdb
Chạy   : python build_data.py [--src data/XAUUSD_M1.parquet] [--root data]
"""
import argparse
import hashlib
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

SCRIPT_VERSION = "1.0"
SOURCE = "mt5"
SYMBOL = "GOLD"
MIN_BARS_FULL_DAY = 1000          # ngày có > 1000 nến mới coi là M1 thật (ngày đủ khoảng 1380)
RUN_LEN = 30                      # M1 thật = chuỗi >= 30 nến liên tiếp cách nhau đúng 1 phút
SERVER_MINUS_NY_HOURS = 7         # GIẢ THUYẾT: giờ server = giờ New York + 7 (GMT+2 đông / GMT+3 hè)
TZ_ASSUMPTION = f"server = America/New_York + {SERVER_MINUS_NY_HOURS}h"


def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def write_partitioned(df: pd.DataFrame, root: Path, overwrite: bool) -> bool:
    """Ghi Parquet chia theo năm. Cột đầu tiên (index cũ) phải là cột thời gian."""
    if root.exists() and any(root.rglob("*.parquet")):
        if not overwrite:
            print(f"[giữ nguyên] {root} đã tồn tại (raw bất biến)")
            return False
        shutil.rmtree(root)
    out = df.reset_index()
    out["year"] = out.iloc[:, 0].dt.year.astype("int16")
    root.mkdir(parents=True, exist_ok=True)
    out.to_parquet(root, partition_cols=["year"], index=False)
    print(f"[đã ghi] {root} ({len(out):,} dòng)")
    return True


def cut_coarse_bars(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Bỏ các nến D1/H1 lẫn vào trước khi M1 thật bắt đầu (cắt chính xác đến từng phút)."""
    per_day = df.groupby(df.index.normalize()).size()
    full = per_day[per_day > MIN_BARS_FULL_DAY]
    if full.empty:
        raise RuntimeError("Không có ngày nào > 1000 nến, dữ liệu không phải M1?")
    pos0 = int(df.index.searchsorted(full.index[0]))
    # Đầu ngày đó vẫn có thể còn nến H1 (vd 00:00, 01:00) trước khi M1 thật bắt đầu:
    # tìm chỗ bắt đầu chuỗi RUN_LEN nến liên tiếp cách nhau đúng 1 phút.
    tail = df.index[pos0:]
    step_ok = pd.Series(np.diff(tail.values) == np.timedelta64(1, "m"))
    run = step_ok.rolling(RUN_LEN).min() == 1
    if not run.any():
        raise RuntimeError("Không tìm thấy chuỗi M1 liên tiếp")
    start_pos = pos0 + int(run.idxmax()) - (RUN_LEN - 1)
    n_cut = start_pos
    print(f"M1 thật bắt đầu từ {df.index[start_pos]} (giờ server); loại {n_cut:,} nến thô trước đó")
    return df.iloc[start_pos:], n_cut


def to_utc(m1: pd.DataFrame) -> pd.DataFrame:
    ny = (m1.index - pd.Timedelta(hours=SERVER_MINUS_NY_HOURS)).tz_localize(
        "America/New_York", ambiguous="NaT", nonexistent="shift_forward"
    )
    ok = ~ny.isna()
    n_bad = int((~ok).sum())
    if n_bad:
        print(f"Bỏ {n_bad} nến rơi vào giờ mơ hồ lúc chuyển giờ mùa hè")
    m1, ny = m1[ok], ny[ok]
    clean = m1.copy()
    clean.insert(0, "time_server", m1.index)
    clean.index = pd.DatetimeIndex(ny.tz_convert("UTC").tz_localize(None), name="time_utc")
    clean = clean[~clean.index.duplicated(keep="first")].sort_index()
    return clean


def break_check(clean: pd.DataFrame) -> None:
    """Giờ nghỉ hằng ngày phải rỗng: 22:00 UTC (đông) và 21:00 UTC (hè). In tỷ lệ so với giờ 12:00."""
    print("\n=== KIỂM TRA GIỜ NGHỈ SAU KHI ĐỔI UTC (kỳ vọng gần 0.00) ===")
    print("năm   đông(22h)  hè(21h)")
    for year, g in clean.groupby(clean.index.year):
        def share(months, hour):
            x = g[g.index.month.isin(months)]
            ref = int((x.index.hour == 12).sum())
            return float("nan") if ref == 0 else (x.index.hour == hour).sum() / ref
        w, s = share([12, 1, 2], 22), share([6, 7, 8], 21)
        flag = "  <-- LỆCH, kiểm tra offset" if (w > 0.2 or s > 0.2) else ""
        print(f"{year}  {w:8.2f}  {s:8.2f}{flag}")


def build_h1(clean: pd.DataFrame) -> pd.DataFrame:
    h1 = clean.resample("1h").agg(
        {"open": "first", "high": "max", "low": "min", "close": "last",
         "tick_volume": "sum", "spread": "median"}
    )
    h1["n_m1"] = clean["close"].resample("1h").count()   # số nến M1 trong giờ (phát hiện giờ thiếu dữ liệu)
    return h1[h1["n_m1"] > 0]                            # nhãn thời gian = giờ MỞ nến


def write_meta(root: Path, sha: str, n_raw: int, n_cut: int, clean: pd.DataFrame,
               clean_root: Path, h1_root: Path) -> None:
    con = duckdb.connect(str(root / "meta.duckdb"))
    con.execute("""
        CREATE TABLE IF NOT EXISTS sources (
            loaded_at TIMESTAMP, source VARCHAR, symbol VARCHAR, tz_assumption VARCHAR,
            file_sha256 VARCHAR, raw_rows BIGINT, coarse_bars_removed BIGINT, clean_rows BIGINT,
            first_bar_utc TIMESTAMP, last_bar_utc TIMESTAMP, script_version VARCHAR)
    """)
    con.execute(
        "INSERT INTO sources VALUES (?,?,?,?,?,?,?,?,?,?,?)",
        [datetime.now(timezone.utc).replace(tzinfo=None), SOURCE, SYMBOL, TZ_ASSUMPTION, sha,
         n_raw, n_cut, len(clean), clean.index[0].to_pydatetime(), clean.index[-1].to_pydatetime(),
         SCRIPT_VERSION],
    )
    for view, path in (("m1_utc", clean_root), ("h1_utc", h1_root)):
        glob = path.resolve().as_posix() + "/**/*.parquet"
        con.execute(f"CREATE OR REPLACE VIEW {view} AS "
                    f"SELECT * FROM read_parquet('{glob}', hive_partitioning=true)")
    con.close()


def write_splits(config_dir: Path, last_day: str) -> None:
    path = config_dir / "splits.json"
    if path.exists():
        print(f"[giữ nguyên] {path} (đã có, không ghi đè)")
        return
    config_dir.mkdir(parents=True, exist_ok=True)
    splits = {
        "_note": "Sửa nếu cần, rồi COMMIT GIT trước khi chạy bất kỳ baseline nào. Không đổi sau khi thấy kết quả.",
        "development": ["2018-01-01", "2025-06-30"],   # phát triển + walk-forward
        "robustness": ["2014-07-01", "2017-12-31"],    # kiểm tra độ bền bổ sung, không dùng để tinh chỉnh
        "holdout": ["2025-07-01", last_day],           # chạy đúng 1 lần khi cấu hình đã đóng băng
        "timezone": "UTC",
    }
    path.write_text(json.dumps(splits, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"[đã tạo] {path} -> nhớ commit git trước khi chạy baseline")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", default="data/GOLD_M1.parquet")
    ap.add_argument("--root", default="data")
    args, _ = ap.parse_known_args()   # bỏ qua tham số lạ (vd khi chạy trong Jupyter)
    src, root = Path(args.src), Path(args.root)

    raw = pd.read_parquet(src).sort_index()
    raw = raw[~raw.index.duplicated(keep="first")]
    print(f"Đọc {len(raw):,} dòng thô, {raw.index[0]} -> {raw.index[-1]} (giờ server)")

    # 1) RAW: bất biến, thư mục đặt theo hash nên lần tải mới sẽ tạo thư mục mới, không ghi đè
    sha = sha256_of(src)
    raw_root = root / "raw" / SOURCE / f"{SYMBOL}_M1" / sha[:8]
    write_partitioned(raw, raw_root, overwrite=False)

    # 2) CLEAN: cắt nến thô, đổi UTC (có thể tạo lại bất cứ lúc nào từ raw)
    m1, n_cut = cut_coarse_bars(raw)
    clean = to_utc(m1)
    clean_root = root / "clean" / f"{SYMBOL}_M1_utc"
    write_partitioned(clean, clean_root, overwrite=True)
    break_check(clean)

    # 3) DERIVED: H1
    h1 = build_h1(clean)
    h1_root = root / "derived" / f"{SYMBOL}_H1_utc"
    write_partitioned(h1, h1_root, overwrite=True)

    # 4) META + SPLITS
    write_meta(root, sha, len(raw), n_cut, clean, clean_root, h1_root)
    write_splits(Path("config"), str(clean.index[-1].date()))

    print("\nXong. Ví dụ truy vấn:")
    print("  import duckdb; con = duckdb.connect('data/meta.duckdb')")
    print("  con.sql(\"SELECT year, count(*) FROM m1_utc GROUP BY year ORDER BY year\").show()")


if __name__ == "__main__":
    main()

In [ ]:
import duckdb
con = duckdb.connect('data/meta.duckdb')
# 1) hồ sơ theo giờ UTC của 2014-2015
con.sql("""SELECT year, date_part('hour', time_utc) h, count(*) n
           FROM m1_utc WHERE year IN (2014, 2015) GROUP BY ALL ORDER BY year, h""").show(max_rows=60)
# 2) spread theo năm, để chọn giả định chi phí
con.sql("""SELECT year, count(*) n, median(spread) med, quantile_cont(spread, 0.99) p99,
           avg((spread = 0)::INT) share_zero FROM m1_utc GROUP BY year ORDER BY year""").show()